# Шаг 4. Обучаемое ранжирование и итоговый `answer.csv`

К этому моменту у каждого кандидата есть несколько независимых сигналов релевантности:
скоры и ранги BM25, векторного поиска, fusion, скоры cross-encoder'а по двум чанкам.
Финальный шаг - объединить их одной моделью. Это **learning-to-rank (LTR)** - обучаемое
ранжирование: модель учится на размеченных запросах упорядочивать кандидатов так, чтобы
релевантные оказывались выше. Я использую **LambdaMART** (градиентный бустинг деревьев
с ранжирующей функцией потерь, реализация LightGBM) - стандартный и очень сильный выбор
для табличных ранжирующих признаков.

Про важное решение скажу сразу, потому что оно определило финальную архитектуру.
В одной из промежуточных версий я добавил модели признаки-«приоры» - статистику
популярности статей в calibration (как часто статья была правильным ответом). Локально
это дало +0.02 MAP@10, а на публичном тесте - **минус 0.06**: модель выучила частоты
конкретных статей в 500 размеченных запросах, а на тестовых запросах это распределение
другое. Классическое переобучение, поймать которое локальной валидацией почти невозможно -
приор «протекает» через все фолды сразу, ведь он посчитан по всему calibration. Поэтому
в финальной модели **только признаки пары «запрос–статья»**, не зависящие от того, как
часто статья встречалась в разметке.

Запуск: обычный CPU-runtime, ~10-15 минут. GPU не нужен - все тяжёлые скоры уже посчитаны.

## 0. Установка зависимостей

In [1]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--upgrade-strategy", "only-if-needed",
        "lightgbm>=4.3,<5", "scikit-learn>=1.4,<2", "pyarrow", "joblib", "tqdm",
    ])
    print("Зависимости установлены.")
else:
    print("Не Colab: зависимости из requirements.txt.")

Зависимости установлены.


## 1. Google Диск и скоры из шага 3

In [2]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/avito_rag")
else:
    PROJECT_ROOT = Path.cwd().resolve().parent / "avito_rag"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RERANKER_DIR = ARTIFACTS_DIR / "reranker"
FINAL_DIR = ARTIFACTS_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

required = {
    "articles": ARTIFACTS_DIR / "articles_processed.parquet",
    "development": RERANKER_DIR / "bge_reranker_v2_m3_development_fusion_top_50_scores.parquet",
    "holdout": RERANKER_DIR / "bge_reranker_v2_m3_holdout_fusion_top_50_scores.parquet",
    "test": RERANKER_DIR / "bge_reranker_v2_m3_test_fusion_top_50_scores.parquet",
}
missing = [str(p) for p in required.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Сначала выполните ноутбуки 01-03 — не найдены файлы:\n" + "\n".join(missing))
print("PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/avito_rag


In [3]:
from __future__ import annotations

import json
import math
import re
import unicodedata
from typing import Any, Iterable, Sequence

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRanker
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

RANDOM_STATE = 42
TOP_K = 10
N_SPLITS = 5              # число фолдов кросс-валидации на development

LGBM_PARAMS = {
    "objective": "lambdarank",          # ранжирующая функция потерь LambdaMART
    "metric": "map",
    "eval_at": [10],
    "lambdarank_truncation_level": 10,  # оптимизирую именно топ-10 — как в метрике задачи
    "label_gain": [0, 1],
    "n_estimators": 1500,
    "learning_rate": 0.03,
    "num_leaves": 15,                   # неглубокие деревья: данных всего 400 запросов
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}
EARLY_STOPPING_ROUNDS = 80

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 120)

articles = pd.read_parquet(required["articles"])
dev_pairs = pd.read_parquet(required["development"])
holdout_pairs = pd.read_parquet(required["holdout"])
test_pairs = pd.read_parquet(required["test"])


def normalize_ground_truth(value: Any) -> list[int]:
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    if isinstance(value, str):
        return list(dict.fromkeys(int(x) for x in re.findall(r"\d+", value)))
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return list(dict.fromkeys(int(x) for x in value))
    return [int(value)]


for frame in (dev_pairs, holdout_pairs):
    frame["ground_truth"] = frame["ground_truth"].map(normalize_ground_truth)

for frame in (dev_pairs, holdout_pairs, test_pairs):
    frame["query_id"] = frame["query_id"].astype(np.int64)
    frame["article_id"] = frame["article_id"].astype(np.int64)
    frame["candidate_rank"] = frame["candidate_rank"].astype(np.int32)
    frame["chunk_dense_rank"] = frame["chunk_dense_rank"].astype(np.int8)

assert dev_pairs["query_id"].nunique() == 400
assert holdout_pairs["query_id"].nunique() == 100
assert test_pairs["query_id"].nunique() == 500
assert set(dev_pairs["query_id"]).isdisjoint(set(holdout_pairs["query_id"]))

print("development pairs:", dev_pairs.shape)
print("holdout pairs:", holdout_pairs.shape)
print("test pairs:", test_pairs.shape)

development pairs: (37390, 19)
holdout pairs: (9311, 19)
test pairs: (46542, 18)


## 2. Метрики

In [4]:
def unique_top_k(prediction: Iterable[int], k: int) -> list[int]:
    result, seen = [], set()
    for value in prediction:
        article_id = int(value)
        if article_id in seen:
            continue
        seen.add(article_id)
        result.append(article_id)
        if len(result) >= k:
            break
    return result


def average_precision_at_k(ground_truth, prediction, k: int = TOP_K) -> float:
    relevant = set(int(x) for x in ground_truth)
    if not relevant:
        return 0.0
    hits, precision_sum = 0, 0.0
    for rank, article_id in enumerate(unique_top_k(prediction, k), start=1):
        if article_id in relevant:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / min(len(relevant), k)


def map_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    return float(np.mean([average_precision_at_k(gt, pred, k) for gt, pred in zip(ground_truths, predictions)]))


def recall_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    values = []
    for gt, pred in zip(ground_truths, predictions):
        relevant = set(int(x) for x in gt)
        values.append(len(relevant.intersection(pred[:k])) / len(relevant) if relevant else 0.0)
    return float(np.mean(values))


def mrr_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    values = []
    for gt, pred in zip(ground_truths, predictions):
        relevant = set(int(x) for x in gt)
        rank = next((i for i, a in enumerate(pred[:k], start=1) if a in relevant), None)
        values.append(1.0 / rank if rank else 0.0)
    return float(np.mean(values))


def hit_rate_at_k(ground_truths, predictions, k: int = TOP_K) -> float:
    return float(np.mean([bool(set(gt).intersection(pred[:k])) for gt, pred in zip(ground_truths, predictions)]))


def evaluate_rankings(query_table: pd.DataFrame, rankings: list[list[int]]) -> dict[str, float]:
    truths = query_table["ground_truth"].tolist()
    return {
        "MAP@10": map_at_k(truths, rankings),
        "Recall@10": recall_at_k(truths, rankings),
        "MRR@10": mrr_at_k(truths, rankings),
        "HitRate@10": hit_rate_at_k(truths, rankings),
    }


assert math.isclose(average_precision_at_k([1, 2], [1, 3, 2]), (1.0 + 2.0 / 3.0) / 2.0)
print("Метрики готовы.")

Метрики готовы.


## 3. Из пар «запрос-чанк» в кандидатов «запрос–статья»

У каждого кандидата один или два оценённых чанка. Сворачиваю их в одну строку на пару
(запрос, статья): максимум/среднее/минимум/разброс логитов cross-encoder'а плюс
векторные близости обоих чанков. Разброс - тоже сигнал: если один чанк статьи очень
релевантен, а второй нет, это другая ситуация, чем когда оба «тёплые».

In [5]:
def aggregate_candidate_pairs(pairs: pd.DataFrame, *, has_labels: bool) -> pd.DataFrame:
    ordered = pairs.sort_values(["query_id", "article_id", "chunk_dense_rank", "chunk_index"]).copy()
    key_columns = ["query_id", "article_id"]

    base_columns = [
        "query_text", "candidate_rank", "article_title",
        "bm25_score", "dense_score", "fusion_score",
        "bm25_rank", "dense_rank", "fusion_rank",
    ]
    if "split" in ordered.columns:
        base_columns.insert(0, "split")
    if has_labels:
        base_columns.append("ground_truth")

    base = ordered.groupby(key_columns, sort=False, as_index=False).first()[key_columns + base_columns]

    reranker_agg = (
        ordered.groupby(key_columns, sort=False)["reranker_logit"]
        .agg(
            reranker_best_chunk="first",
            reranker_top2_max="max",
            reranker_top2_mean="mean",
            reranker_top2_min="min",
            reranker_top2_std=lambda values: float(np.std(values.to_numpy(dtype=np.float32), ddof=0)),
        )
        .reset_index()
    )
    reranker_agg["reranker_top2_range"] = reranker_agg["reranker_top2_max"] - reranker_agg["reranker_top2_min"]

    chunk_scores = (
        ordered.pivot_table(index=key_columns, columns="chunk_dense_rank", values="chunk_dense_score", aggfunc="first")
        .rename(columns={1: "chunk_dense_top1", 2: "chunk_dense_top2"})
        .reset_index()
    )
    if "chunk_dense_top1" not in chunk_scores:
        raise RuntimeError("Не найден chunk_dense_rank=1.")
    if "chunk_dense_top2" not in chunk_scores:
        chunk_scores["chunk_dense_top2"] = chunk_scores["chunk_dense_top1"]
    chunk_scores["chunk_dense_top2"] = chunk_scores["chunk_dense_top2"].fillna(chunk_scores["chunk_dense_top1"])
    chunk_scores["chunk_dense_mean"] = (chunk_scores["chunk_dense_top1"] + chunk_scores["chunk_dense_top2"]) / 2.0
    chunk_scores["chunk_dense_gap"] = chunk_scores["chunk_dense_top1"] - chunk_scores["chunk_dense_top2"]

    result = (
        base
        .merge(reranker_agg, on=key_columns, how="left", validate="one_to_one")
        .merge(chunk_scores, on=key_columns, how="left", validate="one_to_one")
    )

    if has_labels:
        result["label"] = [
            int(int(article_id) in set(ground_truth))
            for article_id, ground_truth in zip(result["article_id"], result["ground_truth"])
        ]

    assert result.groupby("query_id").size().eq(50).all()
    assert not result.duplicated(key_columns).any()
    return result


dev_candidates = aggregate_candidate_pairs(dev_pairs, has_labels=True)
holdout_candidates = aggregate_candidate_pairs(holdout_pairs, has_labels=True)
test_candidates = aggregate_candidate_pairs(test_pairs, has_labels=False)

print("development:", dev_candidates.shape)
print("holdout:", holdout_candidates.shape)
print("test:", test_candidates.shape)

development: (20000, 24)
holdout: (5000, 24)
test: (25000, 22)


In [6]:
def minmax_by_query(frame: pd.DataFrame, column: str) -> pd.Series:
    minimum = frame.groupby("query_id")[column].transform("min")
    shifted = frame[column] - minimum
    maximum = shifted.groupby(frame["query_id"]).transform("max")
    return shifted.div(maximum.where(maximum > 0, 1.0))


# Zero-shot бейзлайн шага 3 — точка отсчёта для LTR.
for frame in (dev_candidates, holdout_candidates, test_candidates):
    frame["baseline_retrieval_norm"] = minmax_by_query(frame, "fusion_score")
    frame["baseline_reranker_norm"] = minmax_by_query(frame, "reranker_top2_mean")
    frame["baseline_final_score"] = 0.75 * frame["baseline_reranker_norm"] + 0.25 * frame["baseline_retrieval_norm"]


def build_query_table(frame: pd.DataFrame, *, has_labels: bool) -> pd.DataFrame:
    columns = ["query_id", "query_text"]
    if has_labels:
        columns.append("ground_truth")
    return frame[columns].drop_duplicates("query_id").sort_values("query_id").reset_index(drop=True)


dev_query_table = build_query_table(dev_candidates, has_labels=True)
holdout_query_table = build_query_table(holdout_candidates, has_labels=True)
test_query_table = build_query_table(test_candidates, has_labels=False)


def rankings_from_scores(candidate_frame: pd.DataFrame, score_column: str, query_order: Sequence[int]) -> list[list[int]]:
    ranking_map = {}
    for query_id, group in candidate_frame.groupby("query_id", sort=False):
        ordered = group.sort_values([score_column, "article_id"], ascending=[False, True])
        ranking_map[int(query_id)] = unique_top_k(ordered["article_id"].astype(int).tolist(), TOP_K)
    return [ranking_map[int(query_id)] for query_id in query_order]


baseline_metrics = pd.DataFrame({
    "development": evaluate_rankings(
        dev_query_table,
        rankings_from_scores(dev_candidates, "baseline_final_score", dev_query_table["query_id"].tolist()),
    ),
    "holdout": evaluate_rankings(
        holdout_query_table,
        rankings_from_scores(holdout_candidates, "baseline_final_score", holdout_query_table["query_id"].tolist()),
    ),
}).T
print("Zero-shot бейзлайн (шаг 3):")
display(baseline_metrics.round(4))

Zero-shot бейзлайн (шаг 3):


,MAP@10,Recall@10,MRR@10,HitRate@10
development,0.5501,0.8973,0.5935,0.94
holdout,0.6480,0.9450,0.6756,0.96


## 4. Признаки

Два блока признаков, всего 62:

- **Скоровые** - всё, что уже посчитано предыдущими ступенями: скоры и ранги BM25 / dense /
  fusion, агрегаты логитов cross-encoder'а, векторные близости чанков. Плюс их производные:
  обратные и логарифмированные ранги, per-query min-max и z-score нормировки (модель видит
  не только «скор 0.83», но и «насколько этот кандидат выделяется среди своих 50»), разности
  и произведения сигналов разных ступеней.
- **Лексико-статические** - прямые совпадения запроса со статьёй: сколько общих слов
  с заголовком и телом, коэффициент Жаккара (доля общих слов среди всех), точное вхождение
  заголовка в запрос и наоборот, длины текстов.

Чего здесь сознательно **нет**: никакой статистики популярности статей в calibration
(см. историю с приором во введении). Каждый признак описывает пару «запрос–статья»
и переносится на любые новые запросы.

In [7]:
TOKEN_RE = re.compile(r"<[A-Z][A-Z0-9_]*>|[A-Za-zА-Яа-яЁё0-9]+", re.UNICODE)


def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value)).replace("\u00A0", " ").lower()
    return re.sub(r"\s+", " ", text).strip()


def lexical_tokens(value: Any) -> list[str]:
    return TOKEN_RE.findall(normalize_text(value))


def safe_jaccard(left: set[str], right: set[str]) -> float:
    union = left | right
    return len(left & right) / len(union) if union else 0.0


articles_features = articles[["article_id", "clean_title", "clean_body", "document_token_count"]].copy()
articles_features["title_norm"] = articles_features["clean_title"].map(normalize_text)
articles_features["title_tokens"] = articles_features["clean_title"].map(lambda v: frozenset(lexical_tokens(v)))
articles_features["body_tokens"] = articles_features["clean_body"].map(lambda v: frozenset(lexical_tokens(v)))
articles_features["title_token_count"] = articles_features["title_tokens"].map(len)
articles_features["body_token_count"] = articles_features["body_tokens"].map(len)
article_feature_map = articles_features.set_index("article_id").to_dict("index")


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    query_cache = {}
    for query_id, query_text in result[["query_id", "query_text"]].drop_duplicates("query_id").itertuples(index=False):
        query_cache[int(query_id)] = (normalize_text(query_text), frozenset(lexical_tokens(query_text)))

    lexical_rows = []
    for row in tqdm(result[["query_id", "article_id"]].itertuples(index=False), total=len(result), desc="Лексические признаки"):
        query_norm, query_tokens = query_cache[int(row.query_id)]
        article_data = article_feature_map[int(row.article_id)]

        query_set = set(query_tokens)
        title_norm = article_data["title_norm"]
        title_tokens = set(article_data["title_tokens"])
        body_tokens = set(article_data["body_tokens"])
        title_shared = query_set & title_tokens
        body_shared = query_set & body_tokens

        lexical_rows.append({
            "query_char_count": len(query_norm),
            "query_token_count": len(query_set),
            "title_token_count": article_data["title_token_count"],
            "body_token_count": article_data["body_token_count"],
            "document_token_count": article_data["document_token_count"],
            "log_document_token_count": np.log1p(article_data["document_token_count"]),
            "title_shared_count": len(title_shared),
            "title_query_coverage": len(title_shared) / len(query_set) if query_set else 0.0,
            "title_jaccard": safe_jaccard(query_set, title_tokens),
            "body_shared_count": len(body_shared),
            "body_query_coverage": len(body_shared) / len(query_set) if query_set else 0.0,
            "body_jaccard": safe_jaccard(query_set, body_tokens),
            "exact_title_query": float(query_norm == title_norm),
            "title_in_query": float(bool(title_norm) and title_norm in query_norm),
            "query_in_title": float(bool(query_norm) and query_norm in title_norm),
        })

    result = pd.concat([result, pd.DataFrame(lexical_rows, index=result.index)], axis=1)

    for rank_column in ["candidate_rank", "bm25_rank", "dense_rank", "fusion_rank"]:
        result[f"reciprocal_{rank_column}"] = 1.0 / result[rank_column]
        result[f"log_{rank_column}"] = np.log1p(result[rank_column])

    score_columns = [
        "bm25_score", "dense_score", "fusion_score",
        "reranker_best_chunk", "reranker_top2_max", "reranker_top2_mean", "reranker_top2_min",
        "chunk_dense_top1", "chunk_dense_top2", "chunk_dense_mean",
    ]
    for score_column in score_columns:
        result[f"{score_column}_minmax"] = minmax_by_query(result, score_column)
        group = result.groupby("query_id")[score_column]
        mean = group.transform("mean")
        std = group.transform("std").fillna(0.0)
        result[f"{score_column}_zscore"] = (result[score_column] - mean) / std.where(std > 0, 1.0)

    result["reranker_minus_fusion_norm"] = result["reranker_top2_mean_minmax"] - result["fusion_score_minmax"]
    result["reranker_times_fusion_norm"] = result["reranker_top2_mean_minmax"] * result["fusion_score_minmax"]
    result["dense_minus_bm25_norm"] = result["dense_score_minmax"] - result["bm25_score_minmax"]

    return result.replace([np.inf, -np.inf], np.nan)


dev_features = add_features(dev_candidates)
holdout_features = add_features(holdout_candidates)
test_features = add_features(test_candidates)

SCORE_FEATURES = [
    "candidate_rank", "bm25_score", "dense_score", "fusion_score",
    "bm25_rank", "dense_rank", "fusion_rank",
    "reranker_best_chunk", "reranker_top2_max", "reranker_top2_mean", "reranker_top2_min",
    "reranker_top2_std", "reranker_top2_range",
    "chunk_dense_top1", "chunk_dense_top2", "chunk_dense_mean", "chunk_dense_gap",
    "reciprocal_candidate_rank", "reciprocal_bm25_rank", "reciprocal_dense_rank", "reciprocal_fusion_rank",
    "log_candidate_rank", "log_bm25_rank", "log_dense_rank", "log_fusion_rank",
    "bm25_score_minmax", "dense_score_minmax", "fusion_score_minmax",
    "reranker_best_chunk_minmax", "reranker_top2_max_minmax", "reranker_top2_mean_minmax", "reranker_top2_min_minmax",
    "chunk_dense_top1_minmax", "chunk_dense_top2_minmax", "chunk_dense_mean_minmax",
    "bm25_score_zscore", "dense_score_zscore", "fusion_score_zscore",
    "reranker_best_chunk_zscore", "reranker_top2_max_zscore", "reranker_top2_mean_zscore", "reranker_top2_min_zscore",
    "chunk_dense_top1_zscore", "chunk_dense_top2_zscore", "chunk_dense_mean_zscore",
    "reranker_minus_fusion_norm", "reranker_times_fusion_norm", "dense_minus_bm25_norm",
]
LEXICAL_STATIC_FEATURES = [
    "query_char_count", "query_token_count", "title_token_count", "body_token_count",
    "document_token_count", "log_document_token_count",
    "title_shared_count", "title_query_coverage", "title_jaccard",
    "body_shared_count", "body_query_coverage", "body_jaccard",
    "exact_title_query", "title_in_query", "query_in_title",
]
FEATURE_NAMES = SCORE_FEATURES + LEXICAL_STATIC_FEATURES

assert len(FEATURE_NAMES) == len(set(FEATURE_NAMES))
assert not (set(FEATURE_NAMES) - set(dev_features.columns))
assert not (set(FEATURE_NAMES) - set(test_features.columns))
print("Признаков:", len(FEATURE_NAMES))

Лексические признаки:   0%|          | 0/20000 [00:00<?, ?it/s]

Лексические признаки:   0%|          | 0/5000 [00:00<?, ?it/s]

Лексические признаки:   0%|          | 0/25000 [00:00<?, ?it/s]

Признаков: 63


## 5. Валидация: out-of-fold кросс-валидация на development

Схема оценки, которой я доверяю:

- **OOF (out-of-fold)**: development делится на 5 фолдов по запросам; модель 5 раз обучается
  на 4/5 и предсказывает отложенную 1/5. В итоге каждый запрос оценён моделью, которая его
  не видела. Средний MAP@10 по этим предсказаниям - основная локальная оценка.
- Фолды делю **группами по нормализованному тексту запроса** (дубли не расползаются по
  фолдам) и со **стратификацией** по числу релевантных статей.
- Early stopping: число деревьев выбирается по валидационному фолду, для финальной модели
  беру медиану по фолдам.

In [8]:
def query_stratum(ground_truth: list[int]) -> str:
    count = len(ground_truth)
    if count <= 1:
        return "1"
    if count == 2:
        return "2"
    return "3+"


cv_query_table = dev_query_table.copy()
cv_query_table["stratum"] = cv_query_table["ground_truth"].map(query_stratum)
cv_query_table["normalized_query_group"] = cv_query_table["query_text"].map(normalize_text)

splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_splits = []
for train_idx, valid_idx in splitter.split(
    cv_query_table, y=cv_query_table["stratum"], groups=cv_query_table["normalized_query_group"]
):
    train_query_ids = cv_query_table.iloc[train_idx]["query_id"].to_numpy()
    valid_query_ids = cv_query_table.iloc[valid_idx]["query_id"].to_numpy()
    assert set(train_query_ids).isdisjoint(set(valid_query_ids))
    fold_splits.append((train_query_ids, valid_query_ids))

print("Размеры фолдов:", [(len(t), len(v)) for t, v in fold_splits])


def prepare_rank_data(frame: pd.DataFrame, feature_names: list[str]):
    ordered = frame.sort_values(["query_id", "candidate_rank", "article_id"]).reset_index(drop=True)
    X = ordered[feature_names].astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y = ordered["label"].to_numpy(dtype=np.int32) if "label" in ordered.columns else None
    group = ordered.groupby("query_id", sort=False).size().to_numpy(dtype=np.int32)
    assert int(group.sum()) == len(ordered)
    assert np.all(group == 50)
    return X, y, group, ordered


def train_fold(train_frame: pd.DataFrame, valid_frame: pd.DataFrame):
    X_train, y_train, group_train, _ = prepare_rank_data(train_frame, FEATURE_NAMES)
    X_valid, y_valid, group_valid, valid_ordered = prepare_rank_data(valid_frame, FEATURE_NAMES)

    model = LGBMRanker(**LGBM_PARAMS)
    model.fit(
        X_train, y_train, group=group_train,
        eval_set=[(X_valid, y_valid)], eval_group=[group_valid],
        eval_names=["validation"], eval_metric="map", eval_at=[10],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(period=0)],
    )
    best_iteration = int(model.best_iteration_ or LGBM_PARAMS["n_estimators"])
    valid_ordered = valid_ordered.copy()
    valid_ordered["ltr_score"] = model.predict(X_valid, num_iteration=best_iteration)
    return model, valid_ordered, best_iteration


oof_frames, best_iterations, fold_rows = [], [], []

for fold_index, (train_query_ids, valid_query_ids) in enumerate(fold_splits):
    train_frame = dev_features[dev_features["query_id"].isin(train_query_ids)].copy()
    valid_frame = dev_features[dev_features["query_id"].isin(valid_query_ids)].copy()

    model, valid_predictions, best_iteration = train_fold(train_frame, valid_frame)

    fold_query_table = dev_query_table[dev_query_table["query_id"].isin(valid_query_ids)].sort_values("query_id").reset_index(drop=True)
    fold_rankings = rankings_from_scores(valid_predictions, "ltr_score", fold_query_table["query_id"].tolist())
    metrics = evaluate_rankings(fold_query_table, fold_rankings)

    fold_rows.append({"fold": fold_index, "best_iteration": best_iteration, **metrics})
    oof_frames.append(valid_predictions)
    best_iterations.append(best_iteration)
    print(f"fold={fold_index}: MAP@10={metrics['MAP@10']:.4f}, деревьев={best_iteration}")

oof_predictions = pd.concat(oof_frames, ignore_index=True).sort_values(["query_id", "candidate_rank", "article_id"]).reset_index(drop=True)
assert oof_predictions["query_id"].nunique() == 400

oof_rankings = rankings_from_scores(oof_predictions, "ltr_score", dev_query_table["query_id"].tolist())
oof_metrics = evaluate_rankings(dev_query_table, oof_rankings)

FINAL_N_ESTIMATORS = max(50, int(np.median(best_iterations)))

display(pd.DataFrame(fold_rows).round(4))
print("OOF-оценка LTR:", {k: round(v, 4) for k, v in oof_metrics.items()})
print("Число деревьев для финальной модели:", FINAL_N_ESTIMATORS)

Размеры фолдов: [(320, 80), (320, 80), (320, 80), (320, 80), (320, 80)]


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgb

fold=0: MAP@10=0.6833, деревьев=123


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold=1: MAP@10=0.6817, деревьев=430


...

fold=3: MAP@10=0.6900, деревьев=113


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


fold=4: MAP@10=0.6810, деревьев=108


,fold,best_iteration,MAP@10,Recall@10,MRR@10,HitRate@10
0,0,123,0.6833,0.8896,0.7564,0.9375
1,1,430,0.6817,0.9042,0.7518,0.9250
2,2,202,0.6601,0.9354,0.7351,0.9750
3,3,113,0.6900,0.9229,0.7549,0.9625
4,4,108,0.6810,0.9417,0.7521,0.9625


OOF-оценка LTR: {'MAP@10': 0.6792, 'Recall@10': 0.9187, 'MRR@10': 0.7501, 'HitRate@10': 0.9525}
Число деревьев для финальной модели: 123


## 6. Однократная проверка на holdout

Модель, обученная на всех 400 development-запросах, оценивается на 100 запросах holdout,
которые не участвовали ни в одном решении. Это последняя контрольная точка перед test.

In [9]:
train_params = {**LGBM_PARAMS, "n_estimators": FINAL_N_ESTIMATORS}

X_dev, y_dev, group_dev, _ = prepare_rank_data(dev_features, FEATURE_NAMES)
X_holdout, _, group_holdout, holdout_ordered = prepare_rank_data(holdout_features, FEATURE_NAMES)

development_model = LGBMRanker(**train_params)
development_model.fit(X_dev, y_dev, group=group_dev, callbacks=[lgb.log_evaluation(period=0)])

holdout_ordered = holdout_ordered.copy()
holdout_ordered["ltr_score"] = development_model.predict(X_holdout)

holdout_ltr_metrics = evaluate_rankings(
    holdout_query_table,
    rankings_from_scores(holdout_ordered, "ltr_score", holdout_query_table["query_id"].tolist()),
)

comparison = pd.DataFrame({
    "zero-shot reranker (шаг 3)": baseline_metrics.loc["holdout"],
    "LTR (шаг 4)": holdout_ltr_metrics,
}).T
print("Holdout, 100 запросов:")
display(comparison.round(4))

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Holdout, 100 запросов:


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


,MAP@10,Recall@10,MRR@10,HitRate@10
zero-shot reranker (шаг 3),0.6480,0.945,0.6756,0.96
LTR (шаг 4),0.7171,0.970,0.7483,0.97


## 7. Анализ ошибок

Смотрю, где OOF-модель ошибается: считаю AP@10 каждого запроса и разбираю худшие.
Это тот анализ, который в течение работы и определял мои решения.

In [10]:
oof_query_ap = []
truth_by_query = dict(zip(dev_query_table["query_id"], dev_query_table["ground_truth"]))
text_by_query = dict(zip(dev_query_table["query_id"], dev_query_table["query_text"]))
title_by_article = articles.set_index("article_id")["clean_title"].astype(str).to_dict()

for query_id, ranking in zip(dev_query_table["query_id"], oof_rankings):
    truth = truth_by_query[query_id]
    oof_query_ap.append({
        "query_id": int(query_id),
        "query_text": text_by_query[query_id],
        "n_relevant": len(truth),
        "ap10": average_precision_at_k(truth, ranking),
        "found_in_top10": len(set(truth) & set(ranking)),
    })

oof_query_ap = pd.DataFrame(oof_query_ap)

print("Распределение AP@10 по запросам:")
display(oof_query_ap["ap10"].describe(percentiles=[0.1, 0.25, 0.5, 0.75]).round(3).to_frame())
print("Запросов с AP=0 (ни одной релевантной в топ-10):", int(oof_query_ap["ap10"].eq(0).sum()), "из 400")
print("Средний AP@10 по числу релевантных статей:")
display(oof_query_ap.groupby("n_relevant")["ap10"].agg(["mean", "count"]).round(3))

print("\nХудшие запросы — что модель поставила в топ и что было правильным:")
worst = oof_query_ap.nsmallest(5, "ap10")
for row in worst.itertuples(index=False):
    ranking = oof_rankings[dev_query_table["query_id"].tolist().index(row.query_id)]
    truth = truth_by_query[row.query_id]
    print("=" * 100)
    print(f"query_id={row.query_id} | AP@10={row.ap10:.2f} | «{row.query_text}»")
    print("  Правильные:", [f"{a}: {title_by_article.get(a, '')[:60]}" for a in truth])
    print("  Мой топ-5: ", [f"{a}: {title_by_article.get(a, '')[:60]}" for a in ranking[:5]])

Распределение AP@10 по запросам:


,ap10
count,400.000
mean,0.679
std,0.334
min,0.000
10%,0.177
25%,0.417
50%,0.750
75%,1.000
max,1.000


Запросов с AP=0 (ни одной релевантной в топ-10): 19 из 400
Средний AP@10 по числу релевантных статей:


,mean,count
n_relevant,,
1,0.693,223
2,0.679,146
3,0.576,30
4,0.733,1



Худшие запросы — что модель поставила в топ и что было правильным:
query_id=52 | AP@10=0.00 | «Здравствуйте! Не получается оплатить за объявление балами! Что делать»
  Правильные: ['4395: Бонусы']
  Мой топ-5:  ['4440: Платёж прошёл, но есть проблема', '2942: Не получается разместить бесплатно', '4389: Платёж не прошёл', '4008: Не получается опубликовать объявление', '2220: Ожидает оплаты']
query_id=70 | AP@10=0.00 | «Здравствуйте хотел бы узнать, почему такая сумма большая если носки стоят <MONEY>»
  Правильные: ['1951: Кто оплачивает доставку и сколько она стоит']
  Мой топ-5:  ['2232: Неверная цена', '2866: Как проверить, вернулись ли деньги за доставку', '4525: Рекламным агентствам: финансы', '4384: Баланс для покупок', '4361: Продавцу']
query_id=74 | AP@10=0.00 | «Здравствуйте, мне не отвечает клиент, который сегодня должен забрать заказ. Работа выполнена, а она не отвечает ни на звонки, ни на сообщения»
  Правильные: ['4403: Продавцу — товар не забирают или вернули']
  Мой топ-5

Типы ошибок, которые я вижу в этом разборе (и что я с ними делал по ходу работы):

1. **Путаница близких по теме статей** - например, несколько статей про доставку или
   возврат, различающихся ролью (покупатель/продавец) или сервисом. Векторно они почти
   неразличимы. Именно против этого работают cross-encoder (читает формулировки целиком)
   и лексические признаки (точные совпадения слов с заголовком).
2. **Многоответные запросы**: средний AP@10 заметно падает с ростом числа релевантных
   статей - найти все 3–4 правильных сложнее, чем одну. Это подтвердило выбор MAP-ориентированной
   функции потерь (lambdarank с обрезкой на топ-10) вместо оптимизации только первой позиции.
3. **Сверхкороткие и разговорные запросы** («не приходят деньги») - сигналов мало, спасает
   только смысловая близость. Частично закрыто нормализацией текста и fusion с векторным поиском.
4. Отдельный найденный и **закрытый** тип ошибки - переобучение на частоты статей через
   prior-признаки: локально метрика росла, на тесте падала. Решение - полный отказ от
   признаков популярности (введение к этому ноутбуку).

## 8. Финальная модель и предсказание для test

Для test обучаю модель уже на **всех 500** размеченных запросах (development + holdout).
Число деревьев зафиксировано по кросс-валидации.

In [11]:
all_training_features = pd.concat([dev_features, holdout_features], ignore_index=True)
X_all, y_all, group_all, all_ordered = prepare_rank_data(all_training_features, FEATURE_NAMES)

deploy_model = LGBMRanker(**train_params)
deploy_model.fit(X_all, y_all, group=group_all, callbacks=[lgb.log_evaluation(period=0)])
print("Финальная модель обучена на", all_ordered["query_id"].nunique(), "запросах.")

X_test, _, group_test, test_ordered = prepare_rank_data(test_features, FEATURE_NAMES)
test_ordered = test_ordered.copy()
test_ordered["ltr_score"] = deploy_model.predict(X_test)
assert np.isfinite(test_ordered["ltr_score"]).all()

test_query_order = test_query_table["query_id"].tolist()
test_rankings = rankings_from_scores(test_ordered, "ltr_score", test_query_order)

assert len(test_rankings) == 500
assert all(len(r) == TOP_K and len(set(r)) == TOP_K for r in test_rankings)

# Сохраняю модель и конфигурацию для аудита.
deploy_model.booster_.save_model(str(FINAL_DIR / "ltr_model_final.txt"))
joblib.dump({"booster": deploy_model.booster_, "feature_names": FEATURE_NAMES, "n_estimators": FINAL_N_ESTIMATORS},
            FINAL_DIR / "ltr_model_final.joblib")
(FINAL_DIR / "final_config.json").write_text(json.dumps({
    "model": "LightGBM LambdaMART",
    "features": FEATURE_NAMES,
    "n_estimators": FINAL_N_ESTIMATORS,
    "random_state": RANDOM_STATE,
    "oof_metrics": {k: float(v) for k, v in oof_metrics.items()},
    "holdout_metrics": {k: float(v) for k, v in holdout_ltr_metrics.items()},
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Модель и конфигурация сохранены в", FINAL_DIR)

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Финальная модель обучена на 500 запросах.


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:969: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Модель и конфигурация сохранены в /content/drive/MyDrive/avito_rag/artifacts/final


## 9. `answer.csv` и строгая проверка формата

Формирую единственный итоговый файл и проверяю каждое требование условия: колонки
`query_id, answer`, все 500 запросов без пропусков и повторов, ровно 10 уникальных
существующих `article_id` в каждом ответе, одиночные пробелы.

In [12]:
answer = pd.DataFrame({
    "query_id": test_query_order,
    "answer": [" ".join(str(article_id) for article_id in ranking) for ranking in test_rankings],
})

ANSWER_PATH = PROJECT_ROOT / "answer.csv"
answer.to_csv(ANSWER_PATH, index=False)
answer.to_csv(FINAL_DIR / "answer.csv", index=False)

# Проверка перечитыванием файла с диска — ровно того, что будет отправлено.
article_id_set = set(articles["article_id"].astype(int).tolist())
answer_readback = pd.read_csv(ANSWER_PATH)

assert answer_readback.columns.tolist() == ["query_id", "answer"]
assert len(answer_readback) == 500
assert answer_readback["query_id"].is_unique
assert answer_readback["query_id"].tolist() == [int(q) for q in test_query_order]


def parse_answer_string(value: Any) -> list[int]:
    text = str(value).strip()
    if not re.fullmatch(r"\d+(?: \d+){9}", text):
        raise ValueError(f"Строка answer должна содержать ровно 10 ID через пробел: {value!r}")
    return [int(token) for token in text.split(" ")]


parsed = answer_readback["answer"].map(parse_answer_string)
assert parsed.map(len).eq(TOP_K).all()
assert parsed.map(lambda ids: len(set(ids))).eq(TOP_K).all()
assert parsed.map(lambda ids: set(ids).issubset(article_id_set)).all()

print("answer.csv прошёл все проверки формата.")
print("Файл:", ANSWER_PATH, f"({ANSWER_PATH.stat().st_size:,} байт)")
display(answer.head(5))

if IN_COLAB:
    from google.colab import files
    files.download(str(ANSWER_PATH))

answer.csv прошёл все проверки формата.
Файл: /content/drive/MyDrive/avito_rag/answer.csv (26,908 байт)


,query_id,answer
0,1,4396 4308 4435 1899 2511 2943 1901 4328 4387 4214
1,2,4009 1923 4234 4219 4408 3168 4361 2646 4403 2802
2,3,1909 4234 4286 4396 4361 4400 2065 3148 4308 3888
3,4,4532 4234 4400 4361 2865 2646 4408 2866 4331 4403
4,5,4320 4214 3467 3128 1951 4234 4321 4388 4045 4218


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Итоги

Путь метрики MAP@10 по ступеням пайплайна (на локальной валидации):

| Ступень | development |
|---|---|
| BM25 | ~0.26 |
| Dense-поиск по чанкам (bge-m3) | ~0.49 |
| Fusion 0.75/0.25 | ~0.50 |
| + cross-encoder, zero-shot | ~0.56 |
| + LambdaMART (OOF) | ~0.68 |

Результат на публичном тесте — **0.64**: разрыв с локальной оценкой небольшой и ожидаемый
(0.68 — OOF на 400 запросах, статистический шум такой выборки ±0.02, плюс тестовые запросы
просто другие). Важно, что финальная архитектура выбиралась именно по принципу переносимости:
всё, что «улучшало» только локальную метрику (article prior, бленды со степенями свободы,
сид-ансамбли), из решения исключено.

Итоговый файл: `avito_rag/answer.csv`.